In [1]:
import tpx_pipe_ioc as tpi
import dask.dataframe as dd
import threading
import socket
from pathlib import Path
import numpy as np
import time
import pandas as pd
import tpx3awkward as tpx

In [6]:
print(dir(tpx.processing))

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'cluster', 'cluster_raw_df', 'convert_tpx3_file', 'convert_tpx3_files', 'convert_tpx3_files_parallel', 'corrections', 'decode_tpx3_binary', 'decoding', 'files', 'find_unmatched_tpx3_files', 'pipeline', 'raw_as_numpy', 'schemas']


In [3]:
trigger, out_q = tpi.test_boot()
path = Path.cwd() / "test_data"
files = []
for f in path.iterdir():
    if "tpx" in f.suffix:
        files.append(f)
print(files)

PIPELINE: starting up


PIPELINE: daemon processes deployed
[PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000000.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000001.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000002.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000003.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000004.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000005.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000006.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000007.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000008.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000009.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000010.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000011.tpx3'), PosixPath('/home/mtaka/Docu

In [4]:
def server():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        print("SERVER: starting up")
        sock.bind((tpi.HOST,tpi.SERVAL))
        sock.listen()
        conn, addr = sock.accept()
        print("SERVER: connected, sending files")
        with conn:
            for f in files:
                a = np.fromfile(f, dtype="<u8")
                conn.sendall(a)
        # conn.send("")
        # conn.send("")
        print("SERVER: finished sending files, closing")


In [ ]:
t = threading.Thread(target=server)
t.start()
time.sleep(1)
trigger.set()

SERVER: starting up
PIPELINE: Triggered: connecting...
PIPELINE: connection attempt:  0
SERVER: connected, sending files


writing out sequence  0  to buffer  0
WORKER: claimed sequence item  0  of size  9804624  with tail(?) length  0
WORKER: Processing 1225578 ints
writing out sequence  1  to buffer  1
WORKER: claimed sequence item  1  of size  10485760  with tail(?) length  0
WORKER: Processing 1310720 ints
writing out sequence  2  to buffer  2
WORKER: claimed sequence item  2  of size  10485760  with tail(?) length  0
WORKER: Processing 1310720 ints
writing out sequence  3  to buffer  3
WORKER: claimed sequence item  3  of size  10485760  with tail(?) length  0
WORKER: Processing 1310720 ints
writing out sequence  4  to buffer  4
SERVER: finished sending files, closing
0 packet received... suspecting eot... awaiting next update
writing out sequence  5  to buffer  5


Process tpx_file_worker_1:
Traceback (most recent call last):
  File "/home/mtaka/Documents/tpx_pipeline/.pixi/envs/default/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/home/mtaka/Documents/tpx_pipeline/.pixi/envs/default/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mtaka/Documents/tpx_pipeline/tpx_pipe_ioc.py", line 84, in worker
    clustered_df = tpx.cluster_decoded_df(
                   ^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module 'tpx3awkward.processing' has no attribute 'cluster_decoded_df'
Process tpx_file_worker_0:
Traceback (most recent call last):
  File "/home/mtaka/Documents/tpx_pipeline/.pixi/envs/default/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/home/mtaka/Documents/tpx_pipeline/.pixi/envs/default/lib/python3.14/multiprocessi

In [5]:
res = []
while not out_q.empty():
    ind, frame = out_q.get()
    res.append(frame)

total = pd.concat(res)
print(total)

           x    y  ToT           t  chip
0         89  477  325  8960052400     2
1        172  499  325  8960054477     2
2        112  403  175  8960069804     2
3        111  403  125  8960069811     2
4        168  302  175  8960074012     2
...      ...  ...  ...         ...   ...
1194049  300  394  150  2560551453     1
1194050  300  395  125  2560551470     1
1194051  269  336  300  2560578669     1
1194052  287  411  200  2560584044     1
1194053  292  294  325  2560584797     1

[7171100 rows x 5 columns]


In [ ]:
target  = Path.cwd() / "data"
files = []
for file in target.iterdir():
    if "parquet" in file.suffix:
        files.append(file)
print(len(files))
print(files[:5])
df = dd.read_parquet(files)
print(df.index.size.compute())

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_version', 'cluster_raw_df', 'convert_tpx3_file', 'convert_tpx3_files', 'convert_tpx3_files_parallel', 'find_unmatched_tpx3_files', 'processing']


: 